# 04. 기간별 API 사용량 조회하기

03에서는 요청 1회의 토큰을 확인했다. 이번에는 **일정 기간 동안 조직 전체에서 쓴 양**을 조회한다.

## 먼저 알아둘 것

- **일반 API Key로는 조회할 수 없다.** Usage & Cost API는 관리자용 API(Admin API)에 속해서 **Admin API Key**(`sk-ant-admin01-...`)가 필요하다.
  - 일반 키로 요청하면 `401: The Admin API requires an Admin API key or an organization-scoped API key.`
  - **개인(individual) 계정에서는 Admin API를 쓸 수 없다.** Console의 [Organization 설정](https://platform.claude.com/settings/organization)에서 조직을 만든 뒤, 관리자 권한이 있는 사람이 Admin Key를 발급할 수 있다.
  - Admin Key는 조직 전체를 관리할 수 있는 강력한 키이므로 일반 키보다 더 조심해서 보관한다.
- **Python SDK에 이 기능이 없다.** `client.xxx()` 대신 `requests`로 HTTP 요청을 직접 보낸다.
- **조회는 무료다.** 모델을 실행하지 않으므로 토큰을 쓰지 않는다.
- 요청이 끝나고 **보통 5분 안에** 데이터에 반영된다. (가끔 더 늦을 수 있음)
- 시간은 모두 **UTC 기준**이다. 한국 시간(KST)은 UTC+9이므로, 한국 날짜와 하루 경계가 9시간 어긋난다.

## 두 가지 엔드포인트

| 엔드포인트 | 알려주는 것 | 시간 단위 |
|---|---|---|
| `GET /v1/organizations/usage_report/messages` | **토큰 수** (입력/출력/캐시) | `1m`, `1h`, `1d` |
| `GET /v1/organizations/cost_report` | **금액** (USD) | `1d`만 |

Admin Key가 없어도 **Run All**로 끝까지 실행된다. 3, 4번은 "건너뜁니다"를 출력하고 넘어가고, **5. Admin Key 없이: 직접 기록하기**는 정상 동작한다.

## 0. 준비

`.env`에 Admin Key를 **다른 이름으로** 추가한다. 일반 키(`ANTHROPIC_API_KEY`)와 섞이지 않게 하기 위해서다.

```
ANTHROPIC_API_KEY=sk-ant-...          # 기존: 메시지 요청용
ANTHROPIC_ADMIN_KEY=sk-ant-admin01-... # 추가: 사용량 조회용
```

In [1]:
import os
from collections import defaultdict
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv

load_dotenv()

# SDK가 아닌 requests로 직접 요청하므로, 키를 직접 꺼내서 헤더에 넣어야 한다
ADMIN_KEY = os.getenv("ANTHROPIC_ADMIN_KEY")

BASE_URL = "https://api.anthropic.com/v1/organizations"
HEADERS = {
    "x-api-key": ADMIN_KEY or "",
    "anthropic-version": "2023-06-01",  # API 버전. 모든 요청에 필요 (SDK는 자동으로 넣어줌)
}

if ADMIN_KEY:
    print("Admin Key 로드 완료")
else:
    print("ANTHROPIC_ADMIN_KEY가 없습니다. 3, 4번 조회는 건너뛰고 5번만 동작합니다.")

ANTHROPIC_ADMIN_KEY가 없습니다. 3, 4번 조회는 건너뛰고 5번만 동작합니다.


## 1. 공통 함수: 요청 + 페이지 넘기기

한 번의 응답에 담기는 시간 구간(bucket) 수에는 한도가 있다. (`1d` 기준 기본 7개, 최대 31개)
조회 기간이 더 길면 응답에 `has_more: true`와 `next_page`가 오고, 이 값을 `page`로 넘겨 다음 페이지를 받는다.

> 01의 `client.models.list()`는 이 과정을 SDK가 자동으로 해줬지만, 여기서는 직접 반복해야 한다.

In [2]:
def fetch_report(path: str, params: dict) -> list[dict]:
    """Admin API 리포트를 요청하고, 페이지가 여러 개면 끝까지 받아서 bucket 리스트로 반환"""
    params = dict(params)  # 원본 dict를 바꾸지 않도록 복사
    buckets = []

    while True:
        r = requests.get(f"{BASE_URL}/{path}", headers=HEADERS, params=params, timeout=30)
        if r.status_code != 200:
            try:
                message = r.json().get("error", {}).get("message", r.text)
            except ValueError:  # 에러 응답이 JSON이 아닌 경우 (프록시/게이트웨이 오류 등)
                message = r.text
            raise RuntimeError(f"{r.status_code}: {message}")

        body = r.json()
        buckets.extend(body["data"])

        if not body["has_more"]:
            return buckets
        params["page"] = body["next_page"]  # 다음 페이지 요청


def fetch_or_skip(path: str, params: dict) -> list[dict]:
    """fetch_report와 같지만, 실패하면 오류 대신 빈 리스트를 반환
    → Run All을 해도 이 셀에서 멈추지 않고 다음 셀로 넘어간다"""
    if not ADMIN_KEY:
        print("Admin Key가 없어 건너뜁니다.")
        return []
    try:
        return fetch_report(path, params)
    except RuntimeError as e:  # API가 에러를 응답함 (401 권한 없음, 400 잘못된 파라미터 등)
        print(f"조회 실패, 건너뜁니다 → {e}")
    except requests.RequestException as e:  # 서버에 닿지 못함 (네트워크 끊김, 타임아웃 등)
        print(f"네트워크 오류, 건너뜁니다 → {e}")
    return []


def to_rfc3339(dt: datetime) -> str:
    """datetime → API가 요구하는 시간 형식 (예: 2026-10-11T00:00:00Z)"""
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

## 2. 조회 기간 정하기

`starting_at`은 필수, `ending_at`은 선택이다. 아래는 **최근 7일**이다. 기간을 바꾸려면 `days` 값만 바꾸면 된다.
특정 날짜를 지정하려면 `datetime(2026, 10, 1, tzinfo=timezone.utc)`처럼 만든다.

In [3]:
days = 7

ending_at = datetime.now(timezone.utc)
starting_at = (ending_at - timedelta(days=days)).replace(hour=0, minute=0, second=0, microsecond=0)

print(f"조회 기간 (UTC): {to_rfc3339(starting_at)} ~ {to_rfc3339(ending_at)}")

조회 기간 (UTC): 2026-10-11T00:00:00Z ~ 2026-10-18T07:57:57Z


## 3. 토큰 사용량 (usage_report)

`group_by[]=model`로 **모델별**로 나눠서 받는다. 다른 기준(`api_key_id`, `workspace_id` 등)으로도 나눌 수 있다.

응답 구조는 다음과 같다.
```
data: [                                  ← 날짜(bucket)별 리스트
  { starting_at, ending_at,
    results: [                           ← 그 날짜의 모델별 사용량
      { model, uncached_input_tokens, cache_read_input_tokens,
        cache_creation: {...}, output_tokens, ... }
    ] }
]
```
사용량이 없는 날도 `results: []`로 포함된다.

> 03의 `response.usage.input_tokens`와 이름이 다르다. 여기서는 `uncached_input_tokens`이다(캐시를 거치지 않은 입력).

In [4]:
usage_buckets = fetch_or_skip(
    "usage_report/messages",
    {
        "starting_at": to_rfc3339(starting_at),
        "ending_at": to_rfc3339(ending_at),
        "bucket_width": "1d",
        "group_by[]": ["model"],
    },
)


def total_input(r: dict) -> int:
    # 입력 = 캐시 안 거친 입력 + 캐시에서 읽은 입력 + 캐시에 저장한 입력(5분/1시간)
    cache = r["cache_creation"]
    return (
        r["uncached_input_tokens"]
        + r["cache_read_input_tokens"]
        + cache["ephemeral_5m_input_tokens"]
        + cache["ephemeral_1h_input_tokens"]
    )


# 조회에 실패했으면 usage_buckets가 빈 리스트 → 표를 출력하지 않음
if usage_buckets:
    print(f"{'Date (UTC)':<12} {'Model':<26} {'Input':>10} {'Output':>10} {'Total':>10}")
    print("-" * 72)
    for bucket in usage_buckets:
        date = bucket["starting_at"][:10]
        for r in bucket["results"]:
            inp, out = total_input(r), r["output_tokens"]
            print(f"{date:<12} {r['model']:<26} {inp:>10,} {out:>10,} {inp + out:>10,}")

Admin Key가 없어 건너뜁니다.


### 기간 전체 합계 (모델별)

In [5]:
totals = defaultdict(lambda: {"input": 0, "output": 0})

for bucket in usage_buckets:
    for r in bucket["results"]:
        totals[r["model"]]["input"] += total_input(r)
        totals[r["model"]]["output"] += r["output_tokens"]

if usage_buckets:
    print(f"{'Model':<26} {'Input':>12} {'Output':>12} {'Total':>12}")
    print("-" * 65)
    for model, t in sorted(totals.items()):
        print(f"{model:<26} {t['input']:>12,} {t['output']:>12,} {t['input'] + t['output']:>12,}")
else:
    print("조회된 사용량이 없어 건너뜁니다.")

조회된 사용량이 없어 건너뜁니다.


## 4. 비용 (cost_report)

토큰 수가 아니라 **실제 청구 기준 금액**을 알려준다. 03처럼 단가를 곱해 추정할 필요가 없다.

- `amount`는 **센트(cent) 단위의 문자열**이다. `"123.45"` = $1.23 → `float(amount) / 100`
- `group_by[]=description`으로 받으면 항목별(모델, 입력/출력 등)로 나뉘고 `model`, `token_type` 필드가 채워진다.
- 일 단위(`1d`)만 지원한다.

In [6]:
cost_buckets = fetch_or_skip(
    "cost_report",
    {
        "starting_at": to_rfc3339(starting_at),
        "ending_at": to_rfc3339(ending_at),
        "group_by[]": ["description"],
    },
)

cost_by_model = defaultdict(float)
for bucket in cost_buckets:
    for r in bucket["results"]:
        dollars = float(r["amount"]) / 100  # 센트 → 달러
        cost_by_model[r["model"] or r["description"]] += dollars

if cost_buckets:
    print(f"{'Model / 항목':<40} {'Cost (USD)':>12}")
    print("-" * 53)
    for name, dollars in sorted(cost_by_model.items(), key=lambda x: -x[1]):
        print(f"{name:<40} ${dollars:>11.4f}")
    print("-" * 53)
    print(f"{'합계':<40} ${sum(cost_by_model.values()):>11.4f}")

Admin Key가 없어 건너뜁니다.


## 5. Admin Key 없이: 직접 기록하기

Admin Key를 쓸 수 없다면, 요청할 때마다 `response.usage`를 파일에 남겨두고 나중에 기간별로 합산한다.

- 장점: 개인 계정에서도 쓸 수 있고, 요청 단위로 원하는 정보(프롬프트 종류, 실험 이름 등)를 함께 기록할 수 있다.
- 단점: **이 코드로 보낸 요청만** 기록된다. Console의 Playground나 다른 프로그램에서 쓴 양은 빠진다. 정확한 청구 금액은 Console의 [Usage](https://platform.claude.com/usage) / [Cost](https://platform.claude.com/cost) 페이지에서 확인한다.

> 아래 셀은 실제로 메시지를 1회 보내므로 소액의 요금이 발생한다.

In [7]:
import csv
from pathlib import Path

import anthropic

client = anthropic.Anthropic()
LOG_DIR = Path("log")  # 로그 파일을 모아두는 폴더 (노트북 기준 상대 경로)
LOG_FILE = LOG_DIR / "usage_log.csv"
FIELDS = ["timestamp_utc", "model", "input_tokens", "output_tokens", "thinking_tokens"]


def log_usage(response) -> None:
    """응답의 usage를 CSV 파일에 한 줄 추가"""
    u = response.usage
    details = u.output_tokens_details
    row = {
        "timestamp_utc": to_rfc3339(datetime.now(timezone.utc)),
        "model": response.model,  # 요청한 모델이 아니라 실제로 응답한 모델 (fallback 대비)
        "input_tokens": u.input_tokens + (u.cache_creation_input_tokens or 0) + (u.cache_read_input_tokens or 0),
        "output_tokens": u.output_tokens,
        "thinking_tokens": (details.thinking_tokens if details else 0) or 0,
    }
    LOG_DIR.mkdir(exist_ok=True)  # 폴더가 없으면 생성 (이미 있으면 그대로)
    is_new = not LOG_FILE.exists()
    with LOG_FILE.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            writer.writeheader()  # 파일을 처음 만들 때만 헤더 작성
        writer.writerow(row)


# 요청할 때마다 log_usage(response)를 함께 호출
response = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=16000,
    messages=[{"role": "user", "content": "토큰이 뭔지 한 문장으로 설명해줘."}],
)
log_usage(response)
print(f"기록 완료 → {LOG_FILE}")

기록 완료 → usage_log.csv


### 기록한 파일에서 기간별로 합산하기

In [8]:
def summarize_log(start: datetime, end: datetime) -> None:
    if not LOG_FILE.exists():  # 아직 한 번도 기록하지 않았으면 파일이 없음
        print(f"{LOG_FILE} 파일이 없습니다. 위 셀에서 먼저 log_usage()로 기록하세요.")
        return

    totals = defaultdict(lambda: {"requests": 0, "input": 0, "output": 0})

    with LOG_FILE.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ts = datetime.strptime(row["timestamp_utc"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if not (start <= ts < end):
                continue
            t = totals[row["model"]]
            t["requests"] += 1
            t["input"] += int(row["input_tokens"])
            t["output"] += int(row["output_tokens"])

    print(f"{'Model':<26} {'Requests':>9} {'Input':>10} {'Output':>10} {'Total':>10}")
    print("-" * 69)
    for model, t in sorted(totals.items()):
        print(f"{model:<26} {t['requests']:>9} {t['input']:>10,} {t['output']:>10,} {t['input'] + t['output']:>10,}")


summarize_log(starting_at, datetime.now(timezone.utc))

Model                       Requests      Input     Output      Total
---------------------------------------------------------------------
claude-opus-5-5                    1         31        133        164
